# 📓 Moda

> **Meu Caderno Digital · Estatística e Probabilidade para IA e Dados**
> Módulo 01 · Estatística Descritiva › Medidas de Posição › Aula 03

### 🎯 Objetivo da aula:

1. Definir a moda (unimodal, bimodal, multimodal) e explicar por que ela é a **única** medida de posição para dados nominais.
2. Calculá-la **na mão** (dados discretos, agrupados com a fórmula de Czuber) e implementá-la **do zero**, inclusive em **fluxo** com memória constante.
3. Entender a moda como o **pico da densidade** e como o **melhor chute único** sob a perda 0–1 (média: erro quadrático, mediana: erro absoluto, moda: acertar exatamente).
4. Estimar a moda de variáveis **contínuas** (histograma e KDE) e saber por que ela depende de uma escolha.
5. Reconhecer as armadilhas: empates, bins e bandwidth, moda fraca e instável, moda de modas, `NaN` e valores-sentinela, moda que esconde a distribuição.
6. Aplicar a moda em **Análise de Dados, Ciência de Dados, ML, DL e Engenharia**, incluindo o **filtro de moda** para estados de equipamentos.

### 🗺️ Roteiro da aula

| Ato | Seções |
|---|---|
| **I · Por quê** | 1. O problema · 2. A intuição |
| **II · O quê** | 3. Definição formal · 4. Propriedades e relações |
| **III · Como** | 5. Na mão · 6. Em Python · 7. Visualizando |
| **IV · Cuidado** | 8. Armadilhas e limites |
| **V · Onde se aplica** | 9. No trabalho, por área · 10. Mini-caso |
| **VI · Fixar** | 11. Entrevista e exercícios · 12. Resumo e ponte |

> **Legenda dos blocos:** 🧠 Intuição · ⚠️ Armadilha · 🏭 No trabalho · 🎯 Entrevista

In [ ]:
import math
import statistics
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.signal import find_peaks
from scipy.stats import gaussian_kde

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

plt.rcParams.update({
    "figure.dpi": 100, "figure.figsize": (9, 4.2), "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25,
})
COR = {"media": "#d1495b", "mediana": "#00798c", "moda": "#6a4c93", "base": "#30638e", "ponderada": "#edae49"}
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

# O dataset fica em data/, dois níveis acima deste notebook
df = pd.read_csv("../../data/dataset_estatistica.csv")
print(f"Dataset carregado: {df.shape[0]} linhas x {df.shape[1]} colunas")

---
# ATO I · POR QUÊ
## 1 · O problema: quantos pares de cada tamanho comprar?

Você cuida da compra de uma loja de calçados e o orçamento só permite encomendar **um** tamanho em grande quantidade para a vitrine. Para decidir, você registra o tamanho de 400 clientes e calcula as medidas de posição que já conhece:

In [ ]:
tamanhos = np.arange(34, 45)
prob = np.array([0.02, 0.04, 0.08, 0.13, 0.22, 0.17, 0.13, 0.10, 0.06, 0.03, 0.02])
clientes = pd.Series(rng.choice(tamanhos, size=400, p=prob), name="tamanho")

moda = clientes.mode().iloc[0]
print(f"média   = {clientes.mean():.2f}")
print(f"mediana = {clientes.median():.1f}")
print(f"moda    = {moda}   ({(clientes == moda).mean():.1%} dos clientes)")

arredondada = round(clientes.mean())
print(f"\nSe encomendar o tamanho da média arredondada ({arredondada}), atendo {(clientes == arredondada).mean():.1%} dos clientes.")
print(f"Se encomendar o tamanho da moda ({moda}), atendo {(clientes == moda).mean():.1%} dos clientes.")
assert (clientes == moda).mean() > (clientes == arredondada).mean()

A média é uma "medida válida" e mesmo assim **inútil** aqui: nenhuma loja vende tamanho 38,75. A mediana ajuda mais, mas o que você quer saber é outra coisa: **qual tamanho aparece mais**, porque encomendar o tamanho mais frequente é o que atende o maior número de clientes de uma vez.

Essa pergunta ("qual é o valor mais frequente?") leva a duas ideias que guiam a aula:

1. Existe uma medida de posição que funciona até quando **somar não faz sentido** (categorias, cores, marcas, códigos de erro)?
2. Qual é a **regra de decisão** por trás dela, assim como a média minimiza o erro quadrático e a mediana o erro absoluto?

---
## 2 · A intuição: o mais popular

> 🧠 **Intuição 1 · O mais votado.** Cada observação é um voto, e a moda é o **vencedor da eleição** (mesmo sem maioria absoluta). O nome vem daí: "estar na moda" é ser o que mais gente faz ou usa.

> 🧠 **Intuição 2 · O pico da montanha.** Imagine o histograma como uma paisagem. A média é o ponto de equilíbrio, a mediana é o marco que divide a área ao meio, e a moda é o **cume**, o ponto mais alto.

> 🧠 **Intuição 3 · Acertar na mosca.** Se você só ganha ponto quando acerta **exatamente** o valor (e não quando chega perto), o melhor palpite único é o valor mais frequente. Isso fecha a trilogia que vamos explorar na seção 4: a **média** minimiza o erro **quadrático**, a **mediana** o erro **absoluto** e a **moda** a perda **0–1**.

Veja os tamanhos de calçado, com as três medidas marcadas:

In [ ]:
contagem = clientes.value_counts().sort_index()
cores = [COR["moda"] if t == moda else COR["base"] for t in contagem.index]

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.bar(contagem.index, contagem.values, color=cores)
ax.axvline(clientes.mean(), color=COR["media"], ls="--", lw=2, label=f"média = {clientes.mean():.2f}")
ax.axvline(clientes.median(), color=COR["mediana"], ls=":", lw=2.5, label=f"mediana = {clientes.median():.0f}")
ax.set_xticks(contagem.index); ax.set_xlabel("tamanho do calçado"); ax.set_ylabel("nº de clientes")
ax.set_title(f"A barra mais alta é a moda ({moda})"); ax.legend()
plt.tight_layout(); plt.show()

---
# ATO II · O QUÊ
## 3 · Definição formal

**Moda amostral.** Seja $f(v) = \sum_{i=1}^{n} \mathbb{1}[x_i = v]$ a **frequência** do valor $v$. A moda é o valor de maior frequência:

$$\hat{x}_{\text{mo}} = \arg\max_{v}\, f(v)$$

**Moda populacional.** Para uma variável discreta, é o valor mais provável; para uma contínua, é o **pico da função densidade** $f_X$:

$$\text{Mo}(X) = \arg\max_{x}\, P(X = x) \quad \text{(discreta)} \qquad\qquad \text{Mo}(X) = \arg\max_{x}\, f_X(x) \quad \text{(contínua)}$$

**Quantas modas?** Um conjunto pode ter **uma** moda (*unimodal*), **duas** (*bimodal*), várias (*multimodal*) ou nenhuma que se destaque (*amodal*, quando todos os valores têm a mesma frequência).

| Símbolo | Nome | O que é |
|---|---|---|
| $f(v)$ | frequência absoluta | quantas vezes $v$ aparece |
| $\hat{x}_{\text{mo}}$ | moda amostral | estimativa calculada dos dados |
| $\text{Mo}(X)$ | moda populacional | pico da distribuição |
| $\mathbb{1}[\cdot]$ | indicadora | vale 1 se a condição é verdadeira, 0 caso contrário |

### Quatro conexões que valem ouro

1. **A única medida para dados nominais.** Média e mediana exigem somar ou ordenar. Para categorias (cor, marca, cidade, código de erro), **só a moda existe**.
2. **Dados agrupados.** Com frequências por classe, a **fórmula de Czuber** interpola dentro da classe modal:

$$\text{Mo} = L + \frac{\Delta_1}{\Delta_1 + \Delta_2}\, h \qquad \Delta_1 = f_{\text{mo}} - f_{\text{ant}}, \quad \Delta_2 = f_{\text{mo}} - f_{\text{post}}$$

em que $L$ é o limite inferior da classe modal, $h$ sua amplitude, $f_{\text{mo}}$ sua frequência e $f_{\text{ant}}$, $f_{\text{post}}$ as frequências das classes vizinhas.

3. **Perda 0–1.** O valor que minimiza o número de "erros de igualdade" $\sum \mathbb{1}[x_i \neq c]$ é a moda (seção 4, P1).
4. **MAP e classificação.** Em inferência bayesiana, a **moda da posteriori** é a estimativa **MAP**, e um classificador que escolhe a classe mais provável está escolhendo a moda da distribuição prevista.

---
## 4 · Propriedades e relações

Vamos **verificar cada propriedade com código**. Primeiro, duas funções que estimam a moda de uma variável **contínua** (onde "o valor mais frequente" não faz sentido, pois quase nenhum valor se repete) e que usaremos ao longo da aula:

In [ ]:
def moda_histograma(x, bins):
    """Ponto médio da classe mais alta de um histograma."""
    contagem, limites = np.histogram(x, bins=bins)
    k = contagem.argmax()
    return (limites[k] + limites[k + 1]) / 2


def moda_kde(x, bw=None, pontos=1000):
    """Pico da estimativa de densidade por kernel (KDE gaussiano)."""
    kde = gaussian_kde(x, bw_method=bw)
    grade = np.linspace(np.min(x), np.max(x), pontos)
    return grade[np.argmax(kde(grade))]


def picos_kde(x, bw=None, pontos=1000, proeminencia=0.05):
    """Todos os picos relevantes da KDE (para detectar multimodalidade)."""
    kde = gaussian_kde(x, bw_method=bw)
    grade = np.linspace(np.min(x), np.max(x), pontos)
    dens = kde(grade)
    idx, _ = find_peaks(dens, prominence=proeminencia * dens.max())
    return grade[idx], grade, dens

### P1 · A trilogia das perdas: média, mediana e moda

Para qualquer número $c$ usado como "chute único", cada medida minimiza uma **perda** diferente:

$$\bar{x} = \arg\min_c \sum (x_i - c)^2 \qquad \tilde{x} \in \arg\min_c \sum |x_i - c| \qquad \hat{x}_{\text{mo}} = \arg\min_c \sum \mathbb{1}[x_i \neq c]$$

In [ ]:
x = np.array([1, 2, 2, 2, 3, 3, 4, 5, 9, 20])
cs = np.linspace(0, 21, 421)
ci = np.arange(0, 22)

perda_q  = np.array([((x - c) ** 2).sum() for c in cs])
perda_a  = np.array([np.abs(x - c).sum() for c in cs])
perda_01 = np.array([(x != c).sum() for c in ci])

fig, axs = plt.subplots(1, 3, figsize=(14, 3.6))
axs[0].plot(cs, perda_q, color=COR["base"]); axs[0].axvline(x.mean(), color=COR["media"], ls="--", label=f"média = {x.mean():.1f}")
axs[0].set_title("Erro quadrático"); axs[0].legend()
axs[1].plot(cs, perda_a, color=COR["base"]); axs[1].axvline(np.median(x), color=COR["mediana"], ls="--", label=f"mediana = {np.median(x):.1f}")
axs[1].set_title("Erro absoluto"); axs[1].legend()
cores = [COR["moda"] if c == 2 else COR["base"] for c in ci]
axs[2].bar(ci, perda_01, color=cores); axs[2].set_title("Perda 0–1: nº de valores diferentes de c")
for ax in axs: ax.set_xlabel("c (o chute único)")
plt.tight_layout(); plt.show()

assert abs(cs[perda_q.argmin()] - x.mean()) < 0.05
assert abs(cs[perda_a.argmin()] - np.median(x)) < 0.05
assert ci[perda_01.argmin()] == statistics.mode(x.tolist()) == 2
print(f"média = {x.mean():.1f} | mediana = {np.median(x):.1f} | moda = {statistics.mode(x.tolist())}   (três 'centros' diferentes para o mesmo conjunto)")

A forma das três curvas conta a história: a parábola suave (quadrática), o "V" (absoluta) e as **barras** da perda 0–1, que só ligam para acertar **exatamente**. A moda é a barra mais baixa, e as outras duas medidas nem aparecem no gráfico dela.

### P2 · A moda não atravessa transformações (nos dados contínuos)

A mediana obedece a $\text{mediana}(g(x)) = g(\text{mediana}(x))$ para $g$ crescente. A moda **não**: ao transformar uma variável contínua, a densidade é "esticada" de forma desigual e o pico **muda de lugar**. Exemplo clássico: se $X \sim N(0, 0{,}6^2)$, então $Y = e^X$ é lognormal, e

$$\text{Mo}(Y) = e^{-\sigma^2} \neq e^{\text{Mo}(X)} = e^{0} = 1$$

In [ ]:
sigma = 0.6
g = np.random.default_rng(7)
X = g.normal(0, sigma, 20_000)
Y = np.exp(X)

print(f"moda de X = {moda_kde(X):.3f}  ->  exp(moda de X) = {np.exp(moda_kde(X)):.3f}")
print(f"moda de Y = exp(X) estimada = {moda_kde(Y):.3f}   (teoria: exp(-σ²) = {np.exp(-sigma**2):.3f})")
assert abs(moda_kde(Y) - np.exp(-sigma ** 2)) < 0.1
assert abs(moda_kde(Y) - np.exp(moda_kde(X))) > 0.15

> ⚠️ **Nuance.** Em dados **discretos ou categóricos**, qualquer renomeação (bijeção) preserva a moda: se você trocar os nomes das categorias por códigos, a categoria mais frequente continua a mesma. O problema é só nos contínuos, em que a "frequência" é, na verdade, uma densidade.

### P3 · A moda é o estimador mais "nervoso"

Como a moda depende do formato **local** da distribuição (o pico), ela flutua mais de amostra para amostra do que a média e a mediana. Vamos medir isso em amostras lognormais, para as quais conhecemos os valores verdadeiros ($e^{\sigma^2/2}$, $1$ e $e^{-\sigma^2}$):

In [ ]:
verdade = {"média": np.exp(sigma ** 2 / 2), "mediana": 1.0, "moda": np.exp(-sigma ** 2)}
g = np.random.default_rng(1)
estimativas = {"média": [], "mediana": [], "moda": []}
for _ in range(400):
    amostra = g.lognormal(0, sigma, 200)
    estimativas["média"].append(amostra.mean())
    estimativas["mediana"].append(np.median(amostra))
    estimativas["moda"].append(moda_kde(amostra, pontos=400))

nervosismo = pd.DataFrame({
    nome: {"valor verdadeiro": verdade[nome], "média das estimativas": np.mean(v),
           "desvio padrão relativo": np.std(v) / verdade[nome]}
    for nome, v in estimativas.items()
}).T
display(nervosismo.round(4))
assert nervosismo.loc["moda", "desvio padrão relativo"] > nervosismo.loc["mediana", "desvio padrão relativo"]

Com o mesmo tamanho de amostra (200), a **moda** tem o maior desvio relativo. Estimar um **pico** exige mais dados do que estimar um centro, porque só os pontos perto do cume informam, e o resto da amostra pouco ajuda.

### P4 · Média, mediana e moda: a ordem revela a assimetria

Em distribuições **unimodais** e assimétricas, as três medidas se alinham do lado da cauda. Com cauda à direita, a ordem típica é **moda < mediana < média**, e com cauda à esquerda é o inverso. Karl Pearson propôs a relação empírica (aproximada) e dois coeficientes de assimetria:

$$\bar{x} - \hat{x}_{\text{mo}} \approx 3\,(\bar{x} - \tilde{x}) \qquad \text{sk}_1 = \frac{\bar{x} - \hat{x}_{\text{mo}}}{s} \qquad \text{sk}_2 = \frac{3\,(\bar{x} - \tilde{x})}{s}$$

In [ ]:
# Lognormal: as três medidas têm fórmula fechada
mu_l = np.exp(sigma ** 2 / 2); med_l = 1.0; mo_l = np.exp(-sigma ** 2)
print(f"Lognormal (σ=0,6): moda = {mo_l:.3f} < mediana = {med_l:.3f} < média = {mu_l:.3f}")
print(f"  média − moda = {mu_l - mo_l:.3f} | 3·(média − mediana) = {3 * (mu_l - med_l):.3f}   (iguais só aproximadamente)")
assert mo_l < med_l < mu_l

# Renda real
renda = df["renda_mensal"].dropna()
r = renda.to_numpy()
mo_r, md_r, mu_r = moda_kde(r), np.median(r), r.mean()
print(f"\nRenda real: moda (KDE) = {mo_r:.0f} | mediana = {md_r:.0f} | média = {mu_r:.0f}")
print(f"sk1 (Pearson, moda)    = {(mu_r - mo_r) / r.std(ddof=1):.3f}")
print(f"sk2 (Pearson, mediana) = {3 * (mu_r - md_r) / r.std(ddof=1):.3f}")
assert mo_r < md_r < mu_r

A renda segue o padrão clássico: **moda < mediana < média**. Pense na renda típica de uma cidade: o valor mais comum é baixo, o do "meio" é um pouco maior, e a média é puxada para cima pelos poucos rendimentos muito altos.

### Três medidas de posição lado a lado

| | Média | Mediana | Moda |
|---|---|---|---|
| **Exige** | somar e dividir | ordenar | só comparar e contar |
| **Funciona em dados nominais** | não | não | **sim** |
| **Minimiza** | erro quadrático | erro absoluto | perda 0–1 |
| **Robustez a outliers** | nenhuma | alta (50%) | total, mas ignora a magnitude |
| **Estabilidade** | alta | alta | **baixa** (o "pico" flutua) |
| **Unicidade** | única | única (convenção) | **pode haver várias** ou nenhuma |
| **Em contínuas** | direta | direta | depende de bins ou bandwidth |

---
# ATO III · COMO
## 5 · Na mão

**Caso 1: uma moda.** Dados `[3, 5, 3, 7, 3, 5, 8]`. Conte: 3 aparece 3 vezes, 5 aparece 2, 7 e 8 aparecem 1. A moda é **3**.

**Caso 2: bimodal.** Dados `[1, 1, 2, 3, 3, 4]`. Os valores 1 e 3 aparecem 2 vezes cada, e os demais 1 vez. As modas são **1 e 3**.

**Caso 3: amodal.** Dados `[1, 2, 3]`. Todos aparecem uma vez, e todos "empatam". Não há um pico que se destaque.

**Caso 4: dados agrupados.** Rendas de 120 clientes, em classes de R$ mil (os mesmos do exemplo da Mediana):

| Classe | 0–2 | 2–4 | 4–6 | 6–8 | 8–10 |
|---|---|---|---|---|---|
| Frequência $f$ | 20 | 45 | 30 | 15 | 10 |

A classe modal é **2–4** ($f_{\text{mo}} = 45$). Os vizinhos têm $f_{\text{ant}} = 20$ e $f_{\text{post}} = 30$, logo $\Delta_1 = 25$, $\Delta_2 = 15$ e $h = 2$:

$$\text{Mo} = 2 + \frac{25}{25 + 15}\cdot 2 = 2 + 1{,}25 = \mathbf{3{,}25}$$

Agora em Python **puro**, devolvendo **todas** as modas:

In [ ]:
def moda(x):
    """Lista com TODAS as modas (os valores de maior frequência), em Python puro."""
    if len(x) == 0:
        raise ValueError("a lista está vazia")
    contagem = {}
    for v in x:
        contagem[v] = contagem.get(v, 0) + 1
    maior = max(contagem.values())
    return sorted(v for v, c in contagem.items() if c == maior)


def moda_agrupada(limites, freq):
    """Moda de dados agrupados pela fórmula de Czuber (classe modal única)."""
    i = max(range(len(freq)), key=lambda j: freq[j])
    f_ant = freq[i - 1] if i > 0 else 0
    f_post = freq[i + 1] if i < len(freq) - 1 else 0
    d1, d2 = freq[i] - f_ant, freq[i] - f_post
    h = limites[i + 1] - limites[i]
    if d1 + d2 == 0:                                   # platô: usa o ponto médio da classe
        return limites[i] + h / 2
    return limites[i] + d1 / (d1 + d2) * h


print("Caso 1 :", moda([3, 5, 3, 7, 3, 5, 8]))
print("Caso 2 :", moda([1, 1, 2, 3, 3, 4]))
print("Caso 3 :", moda([1, 2, 3]))
print("Caso 4 :", moda_agrupada([0, 2, 4, 6, 8, 10], [20, 45, 30, 15, 10]))
print("Nominal:", moda(["Roupas", "Livros", "Roupas", "Alimentos", "Roupas"]))

In [ ]:
# Testes: precisamos concordar com a biblioteca padrão e falhar com clareza
assert moda([3, 5, 3, 7, 3, 5, 8]) == [3]
assert moda([1, 1, 2, 3, 3, 4]) == [1, 3]
assert moda([1, 2, 3]) == [1, 2, 3]
assert np.isclose(moda_agrupada([0, 2, 4, 6, 8, 10], [20, 45, 30, 15, 10]), 3.25)

for _ in range(300):
    amostra = rng.integers(0, 8, size=rng.integers(1, 40)).tolist()
    assert moda(amostra) == sorted(statistics.multimode(amostra))

try:
    moda([])
except ValueError as erro:
    print("OK  lista vazia ->", erro)

### Bônus de engenharia: o elemento majoritário em um fluxo, com memória constante

Contar as frequências de **todos** os valores exige memória proporcional ao número de valores distintos. Se só queremos saber se existe um valor que aparece em **mais da metade** das observações (uma maioria absoluta), o algoritmo de **Boyer–Moore** descobre o candidato com **uma única variável de contagem**:

In [ ]:
def maioria_boyer_moore(fluxo):
    """Candidato a elemento majoritário (>50%), com memória O(1). Só é a maioria se ela existir."""
    candidato, contador = None, 0
    for x in fluxo:
        if contador == 0:
            candidato, contador = x, 1
        elif x == candidato:
            contador += 1
        else:
            contador -= 1
    return candidato


com_maioria = rng.choice([0, 1, 2], size=100_001, p=[0.60, 0.25, 0.15])
sem_maioria = rng.choice([0, 1, 2], size=100_001, p=[0.40, 0.35, 0.25])
for nome, fluxo in [("com maioria", com_maioria), ("sem maioria", sem_maioria)]:
    cand = maioria_boyer_moore(fluxo)
    freq_real = (fluxo == cand).mean()
    print(f"{nome}: candidato = {cand} | frequência real = {freq_real:.1%} | é maioria (>50%)? {freq_real > 0.5}")
assert maioria_boyer_moore(com_maioria) == 0

O algoritmo **sempre** devolve um candidato, mas só a **segunda passada de verificação** (contar a frequência real) diz se ele é de fato maioria. Na prática ele é a base de contadores de "itens quentes" (*heavy hitters*) em fluxos de dados, e você vai implementar uma generalização no exercício D2.

---
## 6 · Em Python: Pandas, `statistics` e SciPy

A ferramenta mais usada é `Series.mode()`, que **devolve uma `Series`** (porque pode haver várias modas):

In [ ]:
cat = df["categoria_frequente"]
print("pandas  Series.mode()      :", cat.mode().tolist())
print("statistics.mode            :", statistics.mode(cat))
print("statistics.multimode       :", statistics.multimode(cat))
print("Counter.most_common(1)     :", Counter(cat).most_common(1))
print("\nParticipação de cada categoria:")
print(cat.value_counts(normalize=True).round(3).to_string())

In [ ]:
compras = df["compras_ano"]
resultado = stats.mode(compras.to_numpy(), keepdims=False)
print(f"pandas  : moda(s) = {compras.mode().tolist()}")
print(f"scipy   : moda = {resultado.mode} (aparece {resultado.count} vezes, {resultado.count / len(compras):.1%} dos clientes)")
print(f"média = {compras.mean():.2f} | mediana = {compras.median():.1f}   (contagem tipo Poisson: as três ficam próximas)")

### A moda de uma variável contínua é uma **estimativa**

Na renda, quase nenhum valor se repete, então `Series.mode()` devolveria um valor qualquer. Para contínuas é preciso **agrupar** (histograma) ou **suavizar** (KDE), e cada escolha dá um resultado:

In [ ]:
estimativas_moda = {f"histograma ({b} classes)": moda_histograma(r, b) for b in [5, 10, 20, 40]}
estimativas_moda.update({
    "KDE (regra de Scott)":     moda_kde(r),
    "KDE (regra de Silverman)": moda_kde(r, bw="silverman"),
    "KDE (suave, bw = 0,6)":    moda_kde(r, bw=0.6),
    "KDE (fino, bw = 0,15)":    moda_kde(r, bw=0.15),
})
tabela_modas = pd.Series(estimativas_moda, name="moda estimada da renda (R$)")
display(tabela_modas.round(0).to_frame())
print(f"\nAs estimativas vão de R$ {tabela_modas.min():.0f} a R$ {tabela_modas.max():.0f} (amplitude de R$ {tabela_modas.max() - tabela_modas.min():.0f}).")

### Moda por grupo, com a **força** da moda

Uma moda sozinha não diz quão dominante ela é. Acompanhe-a do **número de modas** e da **participação** dela no grupo:

In [ ]:
linhas = {}
for categoria, s in df.groupby("categoria_frequente", observed=True)["compras_ano"]:
    m = s.mode()
    linhas[categoria] = {"n": len(s), "n_modas": len(m), "moda": m.iloc[0], "% na moda": (s == m.iloc[0]).mean()}
resumo_moda = pd.DataFrame.from_dict(linhas, orient="index")
resumo_moda

---
## 7 · Visualizando

A moda é o **pico**, então ela precisa ser vista. Quatro retratos: categorias, contagens discretas, a renda contínua com as três medidas e uma distribuição **bimodal**:

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(13, 8))

# (a) categorias: a barra mais alta
cont = df["categoria_frequente"].value_counts()
axs[0, 0].bar(cont.index, cont.values, color=[COR["moda"] if i == 0 else COR["base"] for i in range(len(cont))])
axs[0, 0].set_title("Variável nominal: só a moda existe"); axs[0, 0].set_ylabel("clientes")

# (b) compras no ano: discreta, com as três medidas
cc = compras.value_counts().sort_index()
axs[0, 1].bar(cc.index, cc.values, color=[COR["moda"] if v == compras.mode().iloc[0] else COR["base"] for v in cc.index])
axs[0, 1].axvline(compras.mean(), color=COR["media"], ls="--", lw=2, label=f"média = {compras.mean():.2f}")
axs[0, 1].axvline(compras.median(), color=COR["mediana"], ls=":", lw=2.5, label=f"mediana = {compras.median():.0f}")
axs[0, 1].set_title("Compras no ano (contagem)"); axs[0, 1].legend()

# (c) renda contínua: histograma + KDE + as três medidas
grade = np.linspace(r.min(), r.max(), 500)
axs[1, 0].hist(r, bins=30, density=True, color=COR["base"], alpha=0.35, edgecolor="white")
axs[1, 0].plot(grade, gaussian_kde(r)(grade), color=COR["base"], lw=2)
axs[1, 0].axvline(mo_r, color=COR["moda"], lw=2.5, label=f"moda (KDE) = {mo_r:.0f}")
axs[1, 0].axvline(md_r, color=COR["mediana"], lw=2.5, ls=":", label=f"mediana = {md_r:.0f}")
axs[1, 0].axvline(mu_r, color=COR["media"], lw=2.5, ls="--", label=f"média = {mu_r:.0f}")
axs[1, 0].set_title("Renda: moda < mediana < média"); axs[1, 0].set_xlabel("R$"); axs[1, 0].legend(fontsize=9)

# (d) bimodal: dois picos
bim = np.concatenate([rng.normal(20, 3, 500), rng.normal(80, 3, 500)])
picos, grade_b, dens_b = picos_kde(bim)
axs[1, 1].fill_between(grade_b, dens_b, color=COR["base"], alpha=0.3)
axs[1, 1].plot(grade_b, dens_b, color=COR["base"], lw=2)
for p in picos:
    axs[1, 1].axvline(p, color=COR["moda"], lw=2)
axs[1, 1].axvline(np.median(bim), color=COR["mediana"], ls=":", lw=2.5, label=f"mediana = {np.median(bim):.0f} (no vale)")
axs[1, 1].set_title(f"Distribuição bimodal: modas em {', '.join(f'{p:.0f}' for p in picos)}"); axs[1, 1].legend()
plt.tight_layout(); plt.show()
assert len(picos) == 2

No retrato **(d)**, a moda faz o que a média e a mediana não conseguem: ela **revela** que existem **duas populações**. A mediana cai no vale entre elas, onde quase não há dados.

---
# ATO IV · CUIDADO
## 8 · Armadilhas e limites

A moda parece a medida mais simples ("o que mais aparece") e é cheia de pegadinhas. Seis armadilhas.

### 8.1 · Em variáveis contínuas, o pico depende do histograma

> ⚠️ Os mesmos 189 valores de renda, com três escolhas de número de classes, geram três "modas":

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3.6), sharey=False)
for ax, b in zip(axs, [6, 20, 60]):
    ax.hist(r, bins=b, color=COR["base"], alpha=0.8, edgecolor="white")
    m = moda_histograma(r, b)
    ax.axvline(m, color=COR["moda"], lw=2.5, label=f"moda = {m:.0f}")
    ax.set_title(f"{b} classes"); ax.set_xlabel("renda (R$)"); ax.legend()
plt.tight_layout(); plt.show()

Nenhuma escolha está "certa". Para contínuas, reporte a **faixa modal** (por exemplo, "a faixa modal fica em torno de R$ 3 mil"), em vez de um número com falsa precisão, e informe como foi estimada.

### 8.2 · Empates: cada biblioteca resolve do seu jeito

> ⚠️ Com duas modas, "a moda" não é um número, e as ferramentas não concordam sobre qual devolver:

In [ ]:
empate = [2, 2, 1, 1, 3]
print("pandas.Series.mode     :", pd.Series(empate).mode().tolist())
print("statistics.mode        :", statistics.mode(empate), "  (a primeira encontrada)")
print("statistics.multimode   :", statistics.multimode(empate))
print("scipy.stats.mode       :", stats.mode(empate, keepdims=False).mode, "  (a menor)")
assert statistics.mode(empate) != stats.mode(empate, keepdims=False).mode

Em código de produção, use `Series.mode()` (ou `multimode`) e **decida explicitamente** o que fazer com o empate, em vez de pegar `iloc[0]` sem pensar.

### 8.3 · Moda fraca e instável

> ⚠️ Quando a moda tem uma participação pequena, ela existe, mas quase não significa nada. Veja a idade dos clientes (valores de 18 a 69, quase uniformes):

In [ ]:
idade = df["idade"]
vc = idade.value_counts()
moda_idade = idade.mode().iloc[0]
print(f"moda(s) = {idade.mode().tolist()} | frequência da moda = {vc.iloc[0]} de {len(idade)} clientes ({vc.iloc[0] / len(idade):.1%})")

g = np.random.default_rng(0)
iguais = 0
for _ in range(1000):
    reamostra = g.choice(idade.to_numpy(), size=len(idade), replace=True)
    valores, cont = np.unique(reamostra, return_counts=True)
    iguais += valores[cont.argmax()] == moda_idade
print(f"em {iguais / 1000:.0%} das 1000 reamostragens (bootstrap) a moda continua sendo {moda_idade}")
assert iguais / 1000 < 0.5

# A moda também é descontínua: uma ou duas observações podem trocá-la
base = pd.Series(["a"] * 10 + ["b"] * 9)
print(f"\nmoda = {base.mode().tolist()} | após +2 observações 'b' -> {pd.concat([base, pd.Series(['b', 'b'])]).mode().tolist()}")

Antes de dar importância a uma moda, olhe **quanto** ela domina (a coluna `% na moda` da seção 6) e quão **estável** ela é em reamostragens (bootstrap).

### 8.4 · Moda de modas

> ⚠️ Como na mediana, a moda geral **não pode ser obtida** a partir das modas dos grupos. Três grupos de clientes, contando as categorias x, y e z:

In [ ]:
contagens = pd.DataFrame({
    "G1": {"x": 4,  "y": 3,  "z": 3},
    "G2": {"x": 4,  "y": 3,  "z": 3},
    "G3": {"x": 30, "y": 40, "z": 30},
})
modas_grupo = contagens.idxmax()
total = contagens.sum(axis=1)

print("moda de cada grupo        :", modas_grupo.tolist())
print("moda das modas dos grupos :", modas_grupo.mode().iloc[0])
print("moda geral (a verdade)    :", total.idxmax(), "|", total.to_dict())
assert modas_grupo.mode().iloc[0] == "x" and total.idxmax() == "y"

Dois grupos pequenos "votam" em **x**, mas o grupo grande (G3) puxa o total para **y**. Para a moda geral você precisa das **contagens**, não das modas.

### 8.5 · `NaN` e valores-sentinela

> ⚠️ Com muitos ausentes, o `NaN` pode ser a "categoria" mais frequente. E valores-sentinela (0, −1, 9999, "N/A") usados para marcar "não informado" viram a moda **sem avisar**:

In [ ]:
s = pd.Series(["a", "a", "b", np.nan, np.nan, np.nan], dtype=object)
print("mode() padrão          :", s.mode().tolist())
print("mode(dropna=False)     :", s.mode(dropna=False).tolist())

renda_suja = pd.Series(np.concatenate([r, np.zeros(60)]))     # 60 cadastros sem renda foram preenchidos com 0
print(f"\nmoda da renda = {renda_suja.mode().iloc[0]} ({(renda_suja == 0).mean():.1%} dos registros)")
assert renda_suja.mode().iloc[0] == 0

Isso tem um lado útil: **uma moda suspeita (0, −1, 9999) é um ótimo detector de dado sujo**. Quando a moda de uma variável numérica for um valor "redondo" ou impossível, investigue antes de usar.

### 8.6 · A moda esconde a distribuição

> ⚠️ A moda só olha a frequência do vencedor, e ignora todo o resto:

In [ ]:
grupo_a = np.array([5] * 40 + [4] * 30 + [3] * 30)
grupo_b = np.array([5] * 40 + [1] * 60)

for nome, g_ in [("A", grupo_a), ("B", grupo_b)]:
    print(f"Grupo {nome}: moda = {statistics.mode(g_.tolist())} | média = {g_.mean():.2f} | % de notas <= 2: {(g_ <= 2).mean():.0%}")

Mesma moda (5), públicos opostos. Use a moda **junto** com a média, a mediana e a dispersão (próxima aula) ou mostre a distribuição inteira.

### Resumo das armadilhas

| Armadilha | Sintoma | Remédio |
|---|---|---|
| Contínuas | a "moda" muda com bins ou bandwidth | reportar a faixa modal e o método |
| Empates | cada biblioteca devolve uma moda | `multimode` e decidir o empate |
| Moda fraca ou instável | % na moda baixo; bootstrap muda | informar a participação e testar a estabilidade |
| Moda de modas | "moda geral" que não bate com os dados | usar as contagens |
| `NaN` e sentinelas | a moda é um 0 ou um `NaN` | `dropna`, investigar modas suspeitas |
| Esconde a distribuição | mesma moda, públicos diferentes | acompanhar de outras medidas e do gráfico |

---
# ATO V · ONDE SE APLICA
## 9 · No trabalho, por área

### 9.1 · Análise de Dados: o que é mais frequente

> 🏭 **No trabalho.** Categoria mais comprada, produto mais vendido, horário de pico, motivo de chamado mais comum, código de erro mais frequente: toda pergunta "qual é o mais comum?" é uma **moda**. Em SQL:

    -- PostgreSQL: moda direta (em empate, escolhe um deles)
    SELECT MODE() WITHIN GROUP (ORDER BY categoria) AS moda FROM clientes;

    -- Snowflake: MODE(categoria)

    -- Qualquer banco: lista TODAS as modas (empates incluídos)
    SELECT categoria, COUNT(*) AS n
    FROM clientes
    GROUP BY categoria
    HAVING COUNT(*) = (SELECT MAX(c) FROM (SELECT COUNT(*) AS c FROM clientes GROUP BY categoria) t);

A tabela de **participação por categoria** (`value_counts(normalize=True)`) é o relatório completo, porque diz **quanto** a moda domina.

### 9.2 · Ciência de Dados: imputação por moda e seu viés

> 🏭 **No trabalho.** Preencher uma variável categórica ausente com a moda é o recurso padrão (`SimpleImputer(strategy="most_frequent")`). Mas ele tem um custo: **infla** a categoria dominante. Vamos medir, apagando 30% da categoria ao acaso:

In [ ]:
from sklearn.impute import SimpleImputer

original = df["categoria_frequente"].astype(object)
com_ausentes = original.mask(rng.random(len(original)) < 0.30)

imputada = SimpleImputer(strategy="most_frequent").fit_transform(com_ausentes.to_numpy().reshape(-1, 1))[:, 0]
comparacao = pd.DataFrame({
    "original":     original.value_counts(normalize=True),
    "após imputar": pd.Series(imputada).value_counts(normalize=True),
}).round(3)
comparacao["diferença"] = comparacao["após imputar"] - comparacao["original"]
display(comparacao)
assert comparacao["diferença"].idxmax() == original.mode().iloc[0]

A categoria mais frequente **cresce** e as demais **encolhem**, e isso distorce qualquer estatística que dependa da distribuição. Alternativas: criar uma categoria **"Ausente"** (muitas vezes o próprio ausente é informativo), imputar com base em outras variáveis ou usar imputação múltipla.

Outro uso da moda: **agrupar categorias raras** em "Outros" antes de fazer *one-hot encoding*, e o *frequency encoding* (trocar a categoria pela sua frequência).

### 9.3 · Machine Learning: baseline, KNN e votação

> 🏭 **No trabalho.** Quatro lugares em que a moda aparece em ML.

**(a) O baseline de classificação.** O classificador mais simples **sempre prevê a classe majoritária** (a moda do rótulo). Sua acurácia é a participação dessa classe, e por isso ela pode **enganar** em dados desbalanceados:

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

y = (rng.random(2000) < 0.05).astype(int)                 # 5% de casos positivos (ex.: fraude)
X0 = np.zeros((len(y), 1))
pred = DummyClassifier(strategy="most_frequent").fit(X0, y).predict(X0)

print(f"acurácia            = {accuracy_score(y, pred):.1%}   <- parece ótima")
print(f"recall (positivos)  = {recall_score(y, pred, zero_division=0):.1%}   <- não detecta nenhum")
print(f"F1 macro            = {f1_score(y, pred, average='macro', zero_division=0):.3f}")
assert np.isclose(accuracy_score(y, pred), 1 - y.mean())

Todo modelo precisa **superar esse baseline** na métrica certa. Acurácia de 95% sem olhar a classe rara é um alerta, e não um mérito.

**(b) O KNN prevê a moda dos vizinhos.** A classificação por $k$ vizinhos é, literalmente, a moda dos rótulos dos $k$ pontos mais próximos. Vamos reimplementar a decisão e conferir com o `scikit-learn`:

In [ ]:
from sklearn.datasets import make_classification
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

Xc, yc = make_classification(n_samples=300, n_features=4, n_informative=3, n_redundant=0, random_state=0)
k = 5
knn = KNeighborsClassifier(n_neighbors=k).fit(Xc, yc)
vizinhos = NearestNeighbors(n_neighbors=k).fit(Xc).kneighbors(Xc, return_distance=False)
pred_manual = (yc[vizinhos].sum(axis=1) > k / 2).astype(int)      # moda dos rótulos dos vizinhos (2 classes, k ímpar)

assert (pred_manual == knn.predict(Xc)).all()
print("KNN = moda dos rótulos dos", k, "vizinhos: a nossa previsão bate com a do scikit-learn nos", len(yc), "pontos.")

**(c) Votação por maioria (*hard voting*).** Em um *ensemble*, a previsão final é a **moda** das previsões dos modelos:

In [ ]:
previsoes = np.array([
    [1, 0, 1, 1, 0],     # modelo A
    [1, 1, 0, 1, 0],     # modelo B
    [0, 0, 1, 1, 1],     # modelo C
])
voto = stats.mode(previsoes, axis=0, keepdims=False).mode
print("previsões finais (moda por coluna):", voto)
assert (voto == (previsoes.sum(axis=0) > 1).astype(int)).all()

**(d) Árvores e florestas.** A previsão de cada folha de uma árvore de classificação é a **classe mais frequente** nela, e a de uma floresta aleatória é a moda das previsões das árvores.

### 9.4 · Deep Learning: *argmax*, temperatura e colapso de modos

> 🏭 **No trabalho.** Uma rede de classificação termina em uma **softmax**, uma distribuição de probabilidade sobre as classes, e a previsão usual (*argmax*) é a **moda** dessa distribuição. Modelos de linguagem fazem o mesmo ao escolher o próximo *token*: a decodificação gulosa (*greedy*) pega a moda, e a **amostragem** sorteia conforme as probabilidades. A **temperatura** $T$ controla o quanto a distribuição é "pontuda":

In [ ]:
logits = np.array([2.0, 1.0, 0.5, 0.1])

def softmax(z, T=1.0):
    e = np.exp((z - z.max()) / T)
    return e / e.sum()

for T in [0.5, 1.0, 2.0]:
    p = softmax(logits, T)
    amostrado = np.bincount(rng.choice(len(p), size=20_000, p=p), minlength=len(p)) / 20_000
    print(f"T = {T}: p = {np.round(p, 3)} | moda (greedy) = {p.argmax()} | amostragem: {np.round(amostrado, 3)}")
    assert p.argmax() == logits.argmax()

A moda (*greedy*) **não muda** com a temperatura, mas a amostragem sim: com $T$ baixo, ela quase sempre devolve a moda; com $T$ alto, explora as outras classes. Já o **colapso de modos** (*mode collapse*) é o defeito típico das GANs: o gerador aprende a reproduzir apenas **um pico** da distribuição real e ignora os demais.

### 9.5 · Engenharia e Indústria: o Pareto das falhas

> 🏭 **No trabalho (manutenção).** Em uma planta, poucos códigos de falha respondem pela maior parte das paradas. A moda é o **primeiro item do Pareto**, ou seja, o ponto de partida de qualquer plano de ação:

In [ ]:
codigos = [f"E{k:02d}" for k in range(1, 13)]
pesos = 1 / np.arange(1, 13); pesos = pesos / pesos.sum()
falhas = pd.Series(rng.choice(codigos, size=2000, p=pesos), name="codigo")

pareto = falhas.value_counts().to_frame("n")
pareto["%"] = pareto["n"] / pareto["n"].sum()
pareto["% acumulado"] = pareto["%"].cumsum()
display(pareto.head(5))
print(f"moda = {falhas.mode().iloc[0]} | {(pareto['% acumulado'] < 0.8).sum() + 1} códigos (de {len(codigos)}) explicam 80% das falhas")
assert falhas.mode().iloc[0] == "E01"

Para limpar **estados categóricos** de telemetria (parada, ocioso, produção, alarme), o análogo do filtro de Hampel é o **filtro de moda**, tema do mini-caso a seguir.

---
## 10 · Mini-caso: limpando o estado operacional de um equipamento

**Contexto.** Um equipamento reporta, a cada amostra, seu **estado**: 0 = parada, 1 = ocioso, 2 = produção, 3 = alarme. O sinal verdadeiro alterna entre estados longos e inclui alguns **alarmes reais e curtos** (4 a 7 amostras). Só que a comunicação falha: cerca de 8% das leituras são trocadas por um estado **aleatório**. O resultado é um sinal que "pisca" (*flicker*) e dispara **alarmes falsos**. Queremos limpar sem perder os alarmes reais.

O **filtro de moda** substitui cada amostra pelo estado **mais frequente** em uma janela de $w$ amostras ao redor dela (uma votação local):

In [ ]:
from scipy.ndimage import label

def simular_estados(seed=3, n=900, p_ruido=0.08):
    g = np.random.default_rng(seed)
    verdadeiro = np.empty(n + 200, dtype=int)
    i, ciclo, k = 0, [0, 1, 2, 1, 2, 0], 0
    while i < n:
        dur = g.integers(60, 140)
        verdadeiro[i:i + dur] = ciclo[k % len(ciclo)]
        i, k = i + dur, k + 1
    verdadeiro = verdadeiro[:n]
    posicoes = np.sort(g.choice(np.arange(30, n - 30, 100), size=6, replace=False))
    duracoes = g.integers(4, 8, size=6)
    eventos = list(zip(posicoes.tolist(), duracoes.tolist()))
    for p, d in eventos:
        verdadeiro[p:p + d] = 3                                     # alarmes reais e curtos
    trocado = (verdadeiro + g.integers(1, 4, n)) % 4                # sempre um estado DIFERENTE do verdadeiro
    obs = np.where(g.random(n) < p_ruido, trocado, verdadeiro)
    return verdadeiro, obs, eventos


def filtro_de_moda(x, w, n_classes=4):
    """Estado mais frequente em janelas de tamanho w (empates: o menor índice)."""
    pad = np.pad(x, w // 2, mode="edge")
    contagens = np.stack([np.convolve(pad == k, np.ones(w, dtype=int), mode="valid") for k in range(n_classes)])
    return contagens.argmax(axis=0)


def alarmes_falsos(est, verdadeiro):
    lab, nb = label(est == 3)
    return sum(not (verdadeiro[lab == i] == 3).any() for i in range(1, nb + 1))


def avaliar_estados(est, verdadeiro, eventos):
    return {
        "acurácia": (est == verdadeiro).mean(),
        "transições": int(np.count_nonzero(np.diff(est))),
        "alarmes reais detectados": sum(bool((est[p:p + d] == 3).any()) for p, d in eventos),
        "alarmes falsos": alarmes_falsos(est, verdadeiro),
    }


verdadeiro, obs, eventos = simular_estados()
janelas = [3, 5, 9, 15, 31]
filtrados = {w: filtro_de_moda(obs, w) for w in janelas}

linhas = {"bruto": avaliar_estados(obs, verdadeiro, eventos)}
linhas.update({f"filtro de moda (w = {w})": avaliar_estados(f, verdadeiro, eventos) for w, f in filtrados.items()})
resultado = pd.DataFrame(linhas).T
resultado["alarmes reais detectados"] = resultado["alarmes reais detectados"].astype(int).astype(str) + f" de {len(eventos)}"
resultado["transições"] = resultado["transições"].astype(int)
resultado["alarmes falsos"] = resultado["alarmes falsos"].astype(int)
display(resultado)

bruto = avaliar_estados(obs, verdadeiro, eventos)
f9 = avaliar_estados(filtrados[9], verdadeiro, eventos)
f5 = avaliar_estados(filtrados[5], verdadeiro, eventos)
f31 = avaliar_estados(filtrados[31], verdadeiro, eventos)
assert f9["acurácia"] > bruto["acurácia"] and f9["transições"] < bruto["transições"]
assert f9["alarmes falsos"] < bruto["alarmes falsos"]
assert f5["alarmes reais detectados"] >= 4
assert f31["alarmes reais detectados"] <= 1

In [ ]:
rotulos = {0: "parada", 1: "ocioso", 2: "produção", 3: "alarme"}
t = np.arange(len(obs))
zoom = slice(0, 400)

fig, axs = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1.7, 1]})
axs[0].plot(t[zoom], obs[zoom], ".", color="#aaaaaa", ms=4, label="observado (com falhas de leitura)")
axs[0].step(t[zoom], verdadeiro[zoom], where="post", color="black", lw=1.5, label="verdadeiro")
axs[0].step(t[zoom], filtrados[9][zoom], where="post", color=COR["moda"], lw=2, ls="--", label="filtro de moda (w = 9)")
axs[0].set_yticks(list(rotulos)); axs[0].set_yticklabels(list(rotulos.values()))
axs[0].set_xlabel("amostra"); axs[0].set_title("Estado do equipamento"); axs[0].legend(fontsize=8, loc="center right")

acur = [avaliar_estados(filtrados[w], verdadeiro, eventos)["acurácia"] for w in janelas]
frac = [avaliar_estados(filtrados[w], verdadeiro, eventos)["alarmes reais detectados"] / len(eventos) for w in janelas]
axs[1].plot(range(len(janelas)), acur, marker="o", color=COR["base"], label="acurácia")
axs[1].plot(range(len(janelas)), frac, marker="s", color=COR["media"], label="fração dos alarmes reais detectados")
axs[1].set_xticks(range(len(janelas))); axs[1].set_xticklabels(janelas)
axs[1].set_xlabel("janela w"); axs[1].set_ylim(-0.05, 1.05); axs[1].set_title("O dilema da janela"); axs[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

Três leituras do resultado:

1. O filtro **elimina a cintilação**: a acurácia sobe, as transições espúrias despencam e os **alarmes falsos** quase desaparecem. É a votação local ignorando leituras isoladas.
2. Há um **preço**: uma janela grande **apaga também os alarmes reais curtos**. Quando o evento ocupa só uma pequena parte da janela, ele **perde a votação** para o estado dominante ao redor e desaparece.
3. Escolher $w$ é escolher **o evento mais curto que você não pode perder**. Neste experimento, uma janela pequena ($w = 5$) preserva todos os alarmes reais e já corta a maior parte das transições espúrias, enquanto $w = 15$ ou mais apaga quase todos os alarmes.

> 🏭 **Regra prática.** Em sistemas de monitoramento, filtros robustos suavizam, e **suavizar é sempre perder resolução**. Defina primeiro a **duração mínima do evento relevante** e só então ajuste a janela.

---
# ATO VI · FIXAR
## 11 · Entrevista e exercícios

### 🎯 Perguntas de entrevista

Tente responder em voz alta antes de abrir cada resposta.

<details><summary><b>1. Em que situação a moda é a única medida de posição possível?</b></summary>

Com variáveis **nominais** (categorias sem ordem natural, como cor, marca ou código de erro): não dá para somar (média) nem ordenar (mediana), mas dá para contar frequências.
</details>

<details><summary><b>2. Que perda cada medida de posição minimiza?</b></summary>

A **média** minimiza o erro quadrático, a **mediana** o erro absoluto e a **moda** a perda 0–1 (número de valores diferentes do chute). Em classificação, o classificador de Bayes (que escolhe a classe mais provável) minimiza a perda 0–1.
</details>

<details><summary><b>3. Por que a moda de uma variável contínua é problemática?</b></summary>

Porque quase nenhum valor se repete, então a "moda" precisa ser **estimada** (histograma ou KDE) e depende de uma escolha (número de classes ou bandwidth). Além disso, ela é mais **instável** que a média e a mediana (converge mais devagar) e **não é invariante** a transformações não lineares.
</details>

<details><summary><b>4. Um classificador que sempre prevê a classe majoritária tem 95% de acurácia. É bom?</b></summary>

Provavelmente não. Se a classe majoritária representa 95% dos dados, esse é o **baseline** da moda: um modelo útil precisa superá-lo, e **em métricas que olhem a classe rara** (recall, F1, AUC-PR), porque a acurácia sozinha esconde que o modelo não detecta nenhum caso positivo.
</details>

<details><summary><b>5. Como achar o elemento majoritário em um fluxo, usando memória constante?</b></summary>

Com o algoritmo de **Boyer–Moore** (votação de maioria): mantém um candidato e um contador, e uma passada de **verificação** confirma se ele realmente passa de 50%. Para os "itens quentes" em geral (frequência acima de $n/k$), usa-se o algoritmo de **Misra–Gries**, com $k-1$ contadores.
</details>

<details><summary><b>6. A moda é invariante a transformações dos dados?</b></summary>

Em dados **discretos e categóricos**, é invariante a qualquer renomeação (bijeção). Em **contínuos**, não: uma transformação não linear deforma a densidade e **move o pico** (a moda da lognormal não é o exponencial da moda da normal).
</details>

### ✏️ Exercícios

Resolva **antes** de olhar o gabarito no apêndice.

**Básico**

- **B1.** Encontre, na mão, as modas de `[4, 7, 4, 9, 7, 1, 4]`, de `[2, 3, 3, 5, 5, 8]` e de `[1, 2, 3]`, e classifique cada conjunto (unimodal, bimodal ou amodal).
- **B2.** Em `categoria_frequente`, quantas categorias são necessárias para cobrir pelo menos 80% dos clientes?
- **B3.** Qual é a moda da `idade`? A que distância da participação "uniforme" ela está? Vale a pena reportá-la?

**Intermediário**

- **I1.** Agrupe a renda em 10 classes e calcule a moda por **Czuber**. Compare com a moda por KDE e com o ponto médio da classe mais alta.
- **I2.** Escreva `moda_ponderada(x, w)`, que soma os pesos por valor e devolve as modas. Aplique em `categoria_frequente` com `peso_avaliacoes` e compare com a moda simples.
- **I3.** Estime, por *bootstrap* (1000 reamostragens), a **probabilidade de a moda da categoria se manter** e a de `compras_ano`. Qual das duas é mais estável?

**Desafio**

- **D1.** Demonstre que a moda minimiza a perda 0–1 e que, na classificação, escolher a classe mais provável minimiza o erro de classificação esperado.
- **D2.** Implemente o algoritmo de **Misra–Gries** (itens quentes em fluxo) com $k$ contadores e verifique que ele retém todo item com frequência maior que $n/(k+1)$.

In [ ]:
# ✏️ Resolva aqui (B1, B2, B3, I1, I2, I3, D2)

---
## 12 · Resumo e ponte

### 🧾 Cheat-sheet

| | Moda |
|---|---|
| **Definição** | valor (ou classe) de maior frequência; em contínuas, o pico da densidade |
| **Pandas** | `s.mode()` (devolve uma `Series`) · `s.value_counts(normalize=True)` |
| **Biblioteca padrão / SciPy** | `statistics.multimode(x)` · `scipy.stats.mode(x)` |
| **Minimiza** | perda 0–1 (acertar exatamente) |
| **Funciona em nominais** | sim, é a única medida de posição |
| **Dados agrupados** | Czuber: $L + \frac{\Delta_1}{\Delta_1+\Delta_2}h$ |
| **Use quando** | dados categóricos, "o mais comum", classe majoritária, detectar populações |
| **Cuidado** | empates · bins e bandwidth · moda fraca · moda de modas · sentinelas |

### 🔑 As 7 ideias para levar

1. A moda é o **valor mais frequente** e a **única** medida de posição para dados nominais.
2. Ela fecha a **trilogia das perdas**: média (quadrática), mediana (absoluta), moda (0–1).
3. Em contínuas, a moda é o **pico da densidade** e precisa ser **estimada** (histograma ou KDE), com resultado que depende da escolha.
4. É a medida mais **instável** e a única que pode ser **múltipla** ou inexistente.
5. Com cauda à direita, vale **moda < mediana < média**, e a ordem revela a assimetria.
6. Uma moda **fraca**, **suspeita** (0, −1, 9999) ou **de modas** engana, então olhe a participação, a estabilidade e as contagens.
7. A moda está na **imputação**, no **baseline** de classificação, no **KNN**, na **votação** de ensembles, na **decodificação gulosa** e nos **filtros de estado**.

### 📚 Referências

- BUSSAB, W. O.; MORETTIN, P. A. *Estatística Básica*. Saraiva.
- MORETTIN, P. A.; SINGER, J. M. *Estatística e Ciência de Dados*. LTC.
- SILVERMAN, B. W. *Density Estimation for Statistics and Data Analysis*. Chapman & Hall.
- BOYER, R. S.; MOORE, J. S. "MJRTY: a fast majority vote algorithm". 1991.
- MISRA, J.; GRIES, D. "Finding repeated elements". *Science of Computer Programming*, 1982.
- Documentação: [`pandas.Series.mode`](https://pandas.pydata.org/docs/reference/api/pandas.Series.mode.html) · [`statistics.multimode`](https://docs.python.org/3/library/statistics.html#statistics.multimode) · [`scipy.stats.gaussian_kde`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.gaussian_kde.html)

### 🌉 Ponte para a próxima aula

Fechamos as **medidas de posição**: a média (e a ponderada), a mediana e a moda respondem à pergunta "**onde está o centro?**". Mas dois conjuntos podem ter o **mesmo centro** e ser completamente diferentes:

In [ ]:
a = rng.normal(50, 2, 5000)
b = rng.normal(50, 20, 5000)
for nome, v in [("A", a), ("B", b)]:
    print(f"Conjunto {nome}: média = {v.mean():5.2f} | mediana = {np.median(v):5.2f} | moda (KDE) = {moda_kde(v):5.2f}")

Mesmas medidas de posição, mundos diferentes. A pergunta que falta é:

> **O quanto os dados se espalham em torno do centro?**

Essa é a missão das **medidas de dispersão** (amplitude, IQR, variância e desvio padrão), tema da próxima aula.

---
# 📎 APÊNDICE · Gabarito comentado

> Só abra depois de tentar resolver sozinho. 😉

In [ ]:
# ── B1 ─────────────────────────────────────────────────────────────
for lista in [[4, 7, 4, 9, 7, 1, 4], [2, 3, 3, 5, 5, 8], [1, 2, 3]]:
    m = moda(lista)
    tipo = "unimodal" if len(m) == 1 else ("bimodal" if len(m) == 2 else "amodal (todas empatam)" if len(m) == len(set(lista)) else "multimodal")
    print(f"B1  {lista} -> moda(s) = {m} ({tipo})")
assert moda([4, 7, 4, 9, 7, 1, 4]) == [4] and moda([2, 3, 3, 5, 5, 8]) == [3, 5]

# ── B2 ─────────────────────────────────────────────────────────────
acumulado = df["categoria_frequente"].value_counts(normalize=True).cumsum()
k80 = int((acumulado < 0.8).sum() + 1)
print(f"\nB2  {k80} categorias cobrem {acumulado.iloc[k80 - 1]:.1%} dos clientes")
assert acumulado.iloc[k80 - 1] >= 0.8

# ── B3 ─────────────────────────────────────────────────────────────
ni = df["idade"].nunique()
vc_i = df["idade"].value_counts()
print(f"\nB3  moda(s) = {df['idade'].mode().tolist()} | participação = {vc_i.iloc[0] / len(df):.1%} | uniforme seria {1 / ni:.1%} (com {ni} idades distintas)")
print(f"    A moda cobre só {vc_i.iloc[0] / len(df):.1%} dos clientes (há {len(df['idade'].mode())} modas empatadas): é fraca e quase não informa nada.")

In [ ]:
# ── I1 ─────────────────────────────────────────────────────────────
freq10, lim10 = np.histogram(r, bins=10)
czuber = moda_agrupada(lim10.tolist(), freq10.tolist())
print(f"I1  Czuber = {czuber:.0f} | KDE = {moda_kde(r):.0f} | ponto médio da classe mais alta = {moda_histograma(r, 10):.0f}")

# ── I2 ─────────────────────────────────────────────────────────────
def moda_ponderada(x, w):
    pesos = {}
    for v, p in zip(x, w):
        pesos[v] = pesos.get(v, 0) + p
    maior = max(pesos.values())
    return sorted(v for v, p in pesos.items() if p == maior)

mp = moda_ponderada(df["categoria_frequente"].tolist(), df["peso_avaliacoes"].tolist())
print(f"\nI2  moda simples = {df['categoria_frequente'].mode().tolist()} | moda ponderada por peso_avaliacoes = {mp}")
assert mp[0] == df.groupby("categoria_frequente")["peso_avaliacoes"].sum().idxmax()

# ── I3 ─────────────────────────────────────────────────────────────
def prob_moda_estavel(valores, B=1000, semente=0):
    g = np.random.default_rng(semente)
    valores = np.asarray(valores)
    u, c = np.unique(valores, return_counts=True)
    moda_orig = u[c.argmax()]
    iguais = 0
    for _ in range(B):
        ru, rc = np.unique(g.choice(valores, size=len(valores), replace=True), return_counts=True)
        iguais += ru[rc.argmax()] == moda_orig
    return iguais / B

p_cat = prob_moda_estavel(df["categoria_frequente"].to_numpy(dtype=str))
p_compras = prob_moda_estavel(df["compras_ano"].to_numpy())
print(f"\nI3  P(moda se mantém): categoria = {p_cat:.0%} | compras_ano = {p_compras:.0%}")
print("    mais estável:", "categoria" if p_cat >= p_compras else "compras_ano")

**D1 · Demonstração.** Para dados: $\sum_{i=1}^{n}\mathbb{1}[x_i \neq c] = n - f(c)$, em que $f(c)$ é a frequência de $c$. Minimizar essa soma é o mesmo que **maximizar $f(c)$**, ou seja, escolher a moda. Para uma variável aleatória discreta, $E\big[\mathbb{1}[X \neq c]\big] = 1 - P(X = c)$, que é mínimo quando $P(X = c)$ é máximo. Na classificação, dado o vetor de atributos $x$, o erro esperado de prever a classe $c$ é $1 - P(Y = c \mid x)$, e ele é mínimo na classe de **maior probabilidade a posteriori**: o **classificador de Bayes** é, portanto, a moda da distribuição condicional (a estimativa MAP). $\blacksquare$

In [ ]:
# ── D2 ─────────────────────────────────────────────────────────────
def misra_gries(fluxo, k):
    """Itens quentes em fluxo com no máximo k contadores. Retém todo item com frequência > n/(k+1)."""
    contadores = {}
    for x in fluxo:
        if x in contadores:
            contadores[x] += 1
        elif len(contadores) < k:
            contadores[x] = 1
        else:
            for chave in list(contadores):
                contadores[chave] -= 1
                if contadores[chave] == 0:
                    del contadores[chave]
    return contadores


itens = np.arange(20)
p_itens = 1 / (itens + 1); p_itens = p_itens / p_itens.sum()
fluxo = rng.choice(itens, size=20_000, p=p_itens)
k = 9
contadores = misra_gries(fluxo.tolist(), k)

reais = Counter(fluxo.tolist())
limiar = len(fluxo) / (k + 1)
quentes = [i for i, c in reais.items() if c > limiar]
print(f"D2  itens com frequência > n/(k+1) = {limiar:.0f}: {sorted(quentes)}")
print(f"    contadores retidos             : {dict(sorted(contadores.items(), key=lambda kv: -kv[1]))}")
assert all(i in contadores for i in quentes)
assert max(contadores, key=contadores.get) == reais.most_common(1)[0][0]